# Imports??

In [1]:
import os
import numpy as np
import ee
import geemap
from dotenv import load_dotenv

load_dotenv()
project_id = os.getenv("GOOGLE_CLOUD_PROJECT_ID") # insert your id here

ee.Authenticate(force=False)
ee.Initialize(project=project_id)


Successfully saved authorization token.


# Helper Functions


In [ ]:
def mask_clouds_landsat(image:ee.Image):
    """Given an Image from the Landsat 8 Collection 2 Tier 1 dataset, applies a mask that removes 
    clouds and shadows."""

    # Select the Quality Assessment band
    qa = image.select('QA_PIXEL')

    # Bits 3 and 4 are Cloud and Cloud Shadow, respectively.
    # We create a mask where these bits are set to 0 (Clear)
    cloud_bit = 1 << 3
    shadow_bit = 1 << 4

    # Combined mask: pixel must have neither cloud nor shadow
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(shadow_bit).eq(0))

    # Apply the mask to the image (making clouds/shadows transparent)
    return image.updateMask(mask)


def add_coords(feature:ee.Feature):
    """Given a Feature, adds the longitude and latitude as properties."""
    return feature.set({
        'longitude': feature.geometry().coordinates().get(0),
        'latitude': feature.geometry().coordinates().get(1)
    })


def get_bands(year: int, month: int, day: int, location:ee.Geometry) -> ee.ImageCollection:
    """Gets the Landsat 8 Collection 2 Tier 1 image bands from (year-1, month, day) to (year, month, day) for the provided geometry."""
    return ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
        .filterDate(ee.Date.fromYMD(year, month, day), ee.Date.fromYMD(year+1, month, day)) \
        .map(mask_clouds_landsat)

def get_indices(img_collection: ee.ImageCollection) -> ee.Image:
    """Given an ImageCollection from the Landsat 8 Collection 2 Tier 1 dataset, creates an image with the median
    Red, Near Infrared, Shortwave Infrared 1 and 2 spectral bands, as well as NDVI, NBR, NDMI indices.
    Learn more about spectral indices here: https://www.geo.university/pages/spectral-indices-in-remote-sensing-and-how-to-interpret-them"""
    # Select original bands
    selected_bands = img_collection.select(['SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']).median()
    # print('Available bands:', img_collection.first().bandNames().getInfo())
    # Calculate indices using normalizedDifference
    ndvi = selected_bands.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI') # NIR - Red (SR_B5 - SR_B4)
    nbr = selected_bands.normalizedDifference(['SR_B5', 'SR_B7']).rename('NBR')   # NIR - SWIR2 (SR_B5 - SR_B7)
    ndmi = selected_bands.normalizedDifference(['SR_B5', 'SR_B6']).rename('NDMI') # NIR - SWIR1 (SR_B5 - SR_B6)

    # Return original bands plus calculated indices
    return selected_bands.addBands([ndvi, nbr, ndmi])


def process_yearly_landsat(year:int, month:int, day:int, base_year:int, base_month:int, base_day:int, location:ee.Geometry)->ee.Image:
    """Given year, month, day and base_year, base_month, base_day, gets bands, indices, and year-to-base_year deltas of the Landsat 8 Collection 2 Tier 1 dataset as per get_indices(), from year to base_year for the provided location (a Geometry)"""
    # gets lagged spectral bands, spectral indices, and deltas from year to base_year
    start_date = ee.Date.fromYMD(year, month, day)


    # Process the target year
    img = get_indices(get_bands(year, month, day, location))
    last_year_img = get_indices(get_bands(base_year, base_month, base_day, location))
    if year != base_year:
        print(year, base_year)
        deltas = img.subtract(last_year_img).rename(img.bandNames().map(lambda n: ee.String(n).cat('_delta')))
        img = img.addBands(deltas)


    # Calculate Lags for labeling - Added .toInt() to avoid '.0' in band names
    lag = ee.Number(base_year).subtract(year)
    lag_suffix = ee.String('_lag').cat(lag.toInt().format())

    # Rename year bands
    img_renamed = img.rename(img.bandNames().map(lambda n: ee.String(n).cat(lag_suffix)))

    return img_renamed \
                .set('year', year) \
                .set('lag', lag) \
                .set('base_year', base_year) \
                .set('system:time_start', start_date.millis())

In [46]:
def get_data(base_year:int, base_month, base_day, location:ee.Geometry, filename:str, numPoints:int)->None:
  """Given a year, a location as a Geometry, a filename, and numPoints, 
     extracts numPoints points from image bands/indices for each target value (whether or not the pixel was deforested in year).
     Then calculates bands and deltas of band up to four years before the base date. Saves to filename."""

  # get targets
  hansen = ee.Image("UMD/hansen/global_forest_change_2025_v1_13")
  lossyear = hansen.select('lossyear')
  datamask = hansen.select('datamask')
  treecover = hansen.select('treecover2000')

  land_forest_mask = datamask.eq(1).And(treecover.gt(50))
  target_band = lossyear.unmask(0).updateMask(land_forest_mask).rename('target')
  valid_loss_mask = target_band.eq(0).Or(target_band.eq(base_year-2000))
  class_band = ee.Image(0).updateMask(valid_loss_mask).where(target_band.eq(base_year-2000), 1).where(target_band.eq(0), 0).rename('class')

  # sample points to include in the dataset
  sampled_points = class_band.stratifiedSample(
    numPoints=numPoints,      # Number of points to sample for each class
    classBand='class',    # The name of the band containing the class labels
    region=location,      # The geographical region to sample within
    scale=30,             # The nominal scale (resolution) in meters at which to sample
    geometries=True,      # Include feature geometries (points) in the output
    seed=0                # A random seed for reproducibility
  )
  sampled_points = sampled_points.map(add_coords)

  experimental = sampled_points.filter(ee.Filter.eq('class', 1))
  control = sampled_points.filter(ee.Filter.eq('class', 0))

  full_set = experimental.merge(control)

  # get features
  years = np.arange(base_year-4, base_year+1).tolist()
  collection = ee.ImageCollection([process_yearly_landsat(y, base_month, base_day, base_year, base_month, base_day, location) for y in years])
 
  # merge features and target for our points
  features_image = collection.toBands()
  targets_with_features = features_image.sampleRegions(
      collection=full_set,
      properties=['class', 'longitude', 'latitude'], # Include the 'class', 'longitude', and 'latitude' properties
      scale=30,
      tileScale=16
  )

  # export dataset
  export_task = ee.batch.Export.table.toDrive(
    collection=targets_with_features,
    description=f'exporting {filename}',
    folder='ee_exports_post_pivot',
    fileNamePrefix=filename,
    fileFormat='CSV'
  )
  export_task.start()

# Generating the Data

In [43]:
# Get three years of training data in the Amazon Rainforest, Siberian Taiga, and Borneo rainforest. Come final model evaluation, I will also extract 2025 as my test set

amazon_roi = ee.Geometry.Rectangle([-75, -15, -50, 5])
siberian_taiga_roi = ee.Geometry.Rectangle([80, 50, 120, 70])
borneo_roi = ee.Geometry.Rectangle([108, -5, 119, 8])


In [47]:
get_data(2022, 1, 1, amazon_roi, 'amazon_train_2022_lag0', 3000)
get_data(2023, 1, 1, amazon_roi, 'amazon_train_2023_lag0', 3000)
get_data(2024, 1, 1, amazon_roi, 'amazon_train_2024_lag0', 3000)
get_data(2022, 1, 1, siberian_taiga_roi, 'taiga_train_2022_lag0', 3000)
get_data(2023, 1,1, siberian_taiga_roi, 'taiga_train_2023_lag0', 3000)
get_data(2024, 1, 1, siberian_taiga_roi, 'taiga_train_2024_lag0', 3000)
get_data(2022, 1, 1, borneo_roi, 'borneo_train_2022_lag0', 3000)
get_data(2023, 1, 1, borneo_roi, 'borneo_train_2023_lag0', 3000)
get_data(2024, 1, 1, borneo_roi, 'borneo_train_2024_lag0', 3000)

2018 2022
2019 2022
2020 2022
2021 2022
2019 2023
2020 2023
2021 2023
2022 2023
2020 2024
2021 2024
2022 2024
2023 2024
2018 2022
2019 2022
2020 2022
2021 2022
2019 2023
2020 2023
2021 2023
2022 2023
2020 2024
2021 2024
2022 2024
2023 2024
2018 2022
2019 2022
2020 2022
2021 2022
2019 2023
2020 2023
2021 2023
2022 2023
2020 2024
2021 2024
2022 2024
2023 2024


In [17]:
# get_data(2025, 1, 1, amazon_roi, 'amazon_test_2025', 3000)
# get_data(2025, 1, 1, siberian_taiga_roi, 'taiga_test_2025', 3000)
# get_data(2025,1, 1, borneo_roi, 'borneo_test_2025', 3000)

In [48]:
# --- REPLACE THESE WITH YOUR ACTUAL SCRIPT VARIABLES ---
year = 2023 
month = 1
day = 1
# Ensure your geometry is structured as [Longitude, Latitude]
borneo_geometry = ee.Geometry.Point([114.0, 4.0]) 
# -------------------------------------------------------

# 1. Base Collection
base_collection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

# 2. Test Date Filter Isolation
date_filtered = base_collection.filterDate(
    ee.Date.fromYMD(year-1, month, day), 
    ee.Date.fromYMD(year, month, day)
)
# We limit to 5000 just in case it tries to count the whole globe
print(f"Images after Date filter (Global): {date_filtered.limit(5000).size().getInfo()}")

# 3. Test Bounds Filter Isolation
bounds_filtered = base_collection.filterBounds(borneo_geometry)
print(f"Images after Bounds filter (All Time): {bounds_filtered.size().getInfo()}")

# 4. Combine Date and Bounds
combined_filtered = date_filtered.filterBounds(borneo_geometry)
print(f"Images after Date + Bounds: {combined_filtered.size().getInfo()}")

Images after Date filter (Global): 5000
Images after Bounds filter (All Time): 262
Images after Date + Bounds: 21


In [18]:
assert False

AssertionError: 

This is a cool visualization, comparing the landsat red band (high values indicate forests) with lossyear from the Hansen dataset. Should be run in Google Colab as otherwise it's very slow.

In [ ]:
hansen = ee.Image("UMD/hansen/global_forest_change_2024_v1_12")

# 3. Create an Interactive Map
Map = geemap.Map()

# 4. Define Visualization Parameters
# 'lossyear' is encoded 1-24; we'll visualize it with a gradient
loss_viz = {
    'bands': ['lossyear'],
    'min': 1,
    'max': 24,
    'palette': ['yellow', 'orange', 'red']
}

# Visualization for first_b30 (red band from 2000) to detect forests
# Green for high values (forest), black for low values
first_b30_forest_viz = {
    'min': 0,
    'max': 255,
    'palette': ['black', 'yellow', 'green']
}

# 5. Add layers to the map
# treecover2000 as a background (grayscale)
Map.addLayer(hansen.select('treecover2000'), {'max': 100}, 'Tree Cover 2000')

# Highlight pixels where loss occurred, colored by year
loss_mask = hansen.select('lossyear').gt(0).updateMask(hansen.select('datamask').eq(1))
Map.addLayer(hansen.select('loss'), {'min':0, 'max':1}, name='Loss')
Map.addLayer(hansen.select('first_b30'), first_b30_forest_viz, 'Landsat Red 2000 (Forest)')
Map.addLayer(hansen.select('lossyear'), loss_viz, 'Year of Forest Loss')

vis_params = {
    'color': 'red',       # Border color
    'fillColor': 'blue',  # Fill color (hex codes or named colors work)
    'width': 3            # Stroke width
}

Map.addLayer(amazon_roi, vis_params, 'Amazon')
Map.addLayer(siberian_taiga_roi, vis_params, 'Siberian')
Map.addLayer(borneo_roi, vis_params, 'Borneo')


# 6. Center the map (Longitude, Latitude, Zoom Level)
Map.setCenter(-62.3, -9.2, 10) # Rondônia, Brazil (High deforestation)
Map

Map(center=[-9.2, -62.3], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…